In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import random

# Configuración inicial
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "ecommerce_generated_images"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Cargar el modelo y configurar el scheduler
scheduler = EulerAncestralDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
pipeline = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, scheduler=scheduler, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

# Deshabilitar el safety checker si es necesario
pipeline.safety_checker = None

# 1. Función para generar imágenes desde texto
def generate_image_from_prompt(prompt, guidance_scale=7.5, steps=50, seed=None):
    """
    Genera una imagen desde un prompt textual.
    """
    if seed is not None:
        generator = torch.manual_seed(seed)
    else:
        generator = torch.Generator(device=DEVICE)
    
    image = pipeline(prompt, guidance_scale=guidance_scale, num_inference_steps=steps, generator=generator).images[0]
    return image

# 2. Función para generar imágenes a partir de otras imágenes
def generate_image_from_image(init_image_path, prompt, strength=0.7, guidance_scale=7.5, steps=50, seed=None):
    """
    Genera una nueva imagen basada en una imagen inicial y un prompt.
    """
    img2img_pipeline = StableDiffusionImg2ImgPipeline.from_pretrained(
        MODEL_ID, scheduler=scheduler, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)
    img2img_pipeline.safety_checker = None
    
    init_image = Image.open(init_image_path).convert("RGB").resize((512, 512))
    
    if seed is not None:
        generator = torch.manual_seed(seed)
    else:
        generator = torch.Generator(device=DEVICE)
    
    image = img2img_pipeline(prompt=prompt, init_image=init_image, strength=strength, guidance_scale=guidance_scale, generator=generator).images[0]
    return image

# 3. Guardar imagen generada
def save_image(image, filename):
    """
    Guarda la imagen en el directorio de salida.
    """
    path = os.path.join(OUTPUT_DIR, filename)
    image.save(path)
    print(f"Imagen guardada en: {path}")

# --- Ejecución de pruebas ---

# Prueba 1: Generar imagen desde texto
prompt_text = "A high-resolution photo of red and black athletic sneakers on a white background, studio lighting"
image1 = generate_image_from_prompt(prompt_text, guidance_scale=8.0, steps=50, seed=123)
save_image(image1, "sneakers_from_prompt.png")

# Prueba 2: Generar imagen a partir de otra imagen
init_image_path = "path_to_base_image.jpg"  # Asegúrate de tener una imagen base en este path
prompt_variation = "The same sneakers but in neon colors with a reflective surface"
try:
    image2 = generate_image_from_image(init_image_path, prompt_variation, strength=0.8, guidance_scale=8.0, steps=50, seed=456)
    save_image(image2, "sneakers_variation_neon.png")
except FileNotFoundError:
    print("Imagen base no encontrada. Asegúrate de colocar una imagen en el path especificado.")

# Prueba 3: Generar varias imágenes con diferentes prompts
prompts = [
    "A minimalist photo of a luxury leather handbag with golden accents, studio lighting, e-commerce",
    "A vibrant 3D render of a smartwatch with futuristic elements on a dark background",
    "A pair of casual sneakers on a reflective surface, professional product photography"
]

for idx, prompt in enumerate(prompts):
    image = generate_image_from_prompt(prompt, guidance_scale=7.5, steps=40, seed=random.randint(0, 10000))
    save_image(image, f"product_image_{idx+1}.png")

# --- Conclusión ---
print("Proceso finalizado. Verifica las imágenes en el directorio de salida.")


c:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by

Imagen guardada en: ecommerce_generated_images\sneakers_from_prompt.png


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  8.14it/s]


Imagen base no encontrada. Asegúrate de colocar una imagen en el path especificado.


100%|██████████| 40/40 [06:53<00:00, 10.33s/it]


Imagen guardada en: ecommerce_generated_images\product_image_1.png


100%|██████████| 40/40 [06:57<00:00, 10.45s/it]


Imagen guardada en: ecommerce_generated_images\product_image_2.png


100%|██████████| 40/40 [06:59<00:00, 10.49s/it]


Imagen guardada en: ecommerce_generated_images\product_image_3.png
Proceso finalizado. Verifica las imágenes en el directorio de salida.
